## 数据预处理

In [ ]:
from pathlib import Path
train_save_dir = Path('/root/autodl-tmp/train_data/')
train_save_dir.mkdir(parents=True,exist_ok=True)

test_save_dir = Path('/root/autodl-tmp/test_data/')
test_save_dir.mkdir(parents=True,exist_ok=True)


train_date = ['2025-11-04','2025-11-05','2025-11-06','2025-11-07','2025-11-08','2025-11-09','2025-11-10',
        '2025-11-11','2025-11-12','2025-11-13','2025-11-14','2025-11-15','2025-11-16','2025-11-17',
        '2025-11-18','2025-11-19','2025-11-20','2025-11-21','2025-11-22','2025-11-23','2025-11-24',
        '2025-11-25','2025-11-26','2025-11-27','2025-11-28','2025-11-29','2025-11-30','2025-12-01',
        '2025-12-02','2025-12-03','2025-12-04','2025-12-05','2025-12-06','2025-12-07',
        ]
test_date  = ['2025-12-08','2025-12-09','2025-12-10','2025-12-11','2025-12-12','2025-12-13',
         '2025-12-14','2025-12-15','2025-12-16','2025-12-17','2025-12-18','2025-12-19',
         '2025-12-20','2025-12-21','2025-12-22','2025-12-23']

#### 预处理数据
from pathlib import Path


from Data_Pipeline.preprocessors.lob_data_process import load_lob_data,fullfill_lob_data,generate_labels
from Data_Pipeline.preprocessors.trade_data_process import process_trade_data
lob_data_dir = '/root/autodl-tmp/ETHUSDT/20levels_parquet'
trade_data_dir = '/root/autodl-tmp/ETHUSDT/trade'
save_dir_list = [train_save_dir,test_save_dir]
date_list = [train_date,test_date]
windows=[100,300,600,3000,6000]

for save_dir,date in zip(save_dir_list,date_list):
        levels = 10
        lob_data = load_lob_data(data_dir=lob_data_dir,date = date,levels=levels)
        lob_data = fullfill_lob_data(lob_data)
        lob_data = generate_labels(lob_data,windows=windows)
        labels_cols = [f'return_label_{idx}' for idx in range(len(windows))]
        labels = lob_data.select(['time_bucket']+labels_cols)

        lob_data = lob_data.drop(labels_cols)

        trade_data = process_trade_data(data_dir=trade_data_dir,date = date,window_ms=100)

        lob_data.write_parquet(save_dir / 'lob_data_100ms.parquet')
        trade_data.write_parquet(save_dir / 'trade_data_100ms.parquet')
        labels.write_parquet(save_dir / 'labels_100ms.parquet')

